# 06.2 — Scaled Dot-Product Attention

**The core operation of every Transformer.** Query, Key, Value is the abstraction.

- **Query (Q):** What am I looking for?
- **Key (K):** What do I have?
- **Value (V):** What I return when matched

**Formula:** Attention(Q, K, V) = softmax(QK^T / √d_k) × V

**Why scale by √d_k:** Without scaling, dot products grow with d_k, pushing softmax into saturation regions where gradients are tiny.

---

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: [batch, heads, seq_q, d_k]
    K: [batch, heads, seq_k, d_k]
    V: [batch, heads, seq_k, d_v]
    mask: optional boolean mask [batch, 1, seq_q, seq_k]
    
    Returns:
      output:       [batch, heads, seq_q, d_v]
      attn_weights: [batch, heads, seq_q, seq_k]
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores
    # Q @ K^T: each query dot-products with all keys
    # [batch, heads, seq_q, seq_k]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 2: Apply mask (for padding or causal masking)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Softmax to get attention weights
    # Each row sums to 1: position i attends to all positions
    attn_weights = F.softmax(scores, dim=-1)
    
    # Handle cases where entire row is -inf (all masked)
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
    
    # Step 4: Weighted sum of values
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# Toy example: 4 tokens, d_k = d_v = 8
batch, seq_len, d_k, d_v = 1, 4, 8, 8
Q = torch.randn(batch, 1, seq_len, d_k)
K = torch.randn(batch, 1, seq_len, d_k)
V = torch.randn(batch, 1, seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Q, K, V: {Q.shape}")
print(f"Output:  {output.shape}")
print(f"Weights: {weights.shape}")
print(f"Weights (row sums = 1): {weights[0,0].sum(dim=-1).tolist()}")

In [ ]:
# Why scale by sqrt(d_k)?

print("=== The scaling problem ===")
print()

for d_k in [8, 64, 512]:
    # Random Q and K vectors of dimension d_k
    q = torch.randn(1000, d_k)
    k = torch.randn(1000, d_k)
    
    # Dot products
    dots = (q * k).sum(dim=1)
    
    # Without scaling: variance grows with d_k
    # E[q·k] = 0, Var[q·k] = d_k (when q,k ~ N(0,1))
    unscaled_std = dots.std().item()
    scaled_std = (dots / math.sqrt(d_k)).std().item()
    
    # Softmax gradient: near-zero when inputs have large magnitude
    # (softmax saturates -> one element gets weight ≈1, rest ≈0)
    soft_unscaled = F.softmax(dots.unsqueeze(0), dim=-1)
    max_weight_unscaled = soft_unscaled.max().item()
    
    scaled_dots = dots / math.sqrt(d_k)
    soft_scaled = F.softmax(scaled_dots.unsqueeze(0), dim=-1)
    max_weight_scaled = soft_scaled.max().item()
    
    print(f"d_k={d_k:4d}:  unscaled std={unscaled_std:.2f}  scaled std={scaled_std:.2f}  "
          f"max_attn_weight: unscaled={max_weight_unscaled:.4f}  scaled={max_weight_scaled:.4f}")

print()
print("Without scaling, large d_k -> large dot products -> softmax saturates")
print("Saturated softmax has near-zero gradients -> no learning")
print("Dividing by sqrt(d_k) keeps dot products O(1) variance regardless of dimension")

In [ ]:
# Causal masking: used in decoder (autoregressive generation)
# Position i can only attend to positions 0..i (not future)

def make_causal_mask(seq_len):
    """Lower triangular mask: position i can see positions 0..i."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq, seq]

seq_len = 5
mask = make_causal_mask(seq_len)
print("Causal mask (1=attend, 0=block):")
print(mask[0,0].numpy().astype(int))
print()
print("Row 0: token 0 can only see token 0")
print("Row 2: token 2 can see tokens 0, 1, 2")
print("Row 4: token 4 can see all previous tokens")

# Show masked attention
Q = torch.randn(1, 1, seq_len, 8)
K = torch.randn(1, 1, seq_len, 8)
V = torch.randn(1, 1, seq_len, 8)

_, weights_causal = scaled_dot_product_attention(Q, K, V, mask=mask)
print("\nCausal attention weights:")
print(weights_causal[0,0].detach().numpy().round(3))
print("Upper triangle is 0 (future is masked)")

In [ ]:
# Self-attention vs cross-attention

print("Three uses of attention in a Transformer:")
print()
print("1. ENCODER self-attention:")
print("   Q = K = V = encoder input")
print("   Every position attends to every other (bidirectional)")
print("   No mask")
print()
print("2. DECODER masked self-attention:")
print("   Q = K = V = decoder input")
print("   Position i can only attend to positions 0..i (causal mask)")
print("   Prevents 'cheating' by looking at future tokens during training")
print()
print("3. DECODER cross-attention (encoder-decoder attention):")
print("   Q = decoder hidden, K = V = encoder output")
print("   Each decoder position can attend to all encoder positions")
print("   This is the Bahdanau attention equivalent")
print()
print("In decoder-only models (GPT): only masked self-attention (step 2)")
print("In encoder-only models (BERT): only bidirectional self-attention (step 1)")
print("In encoder-decoder models (T5): all three")

## Summary

**Scaled dot-product attention:** A single differentiable lookup mechanism that can be applied to any sequence.

**Three key design choices:**
1. Dot product similarity (fast, no extra parameters)
2. √d_k scaling (prevents softmax saturation)
3. Masking (controls what each position can see)

**Why this is powerful:** The attention weights are computed from the input, not learned as fixed weights. This means the model can route information differently for each input — dynamic computation.

---
**Next:** 06.3 — Multi-Head Attention